# Visualize High-Support Patterns With Negative Edges

这个 notebook 会：
- 从 `selected_match_counts.csv` 读取每个 pattern 的匹配次数
- 过滤 `match_count >= MIN_SUPPORT`
- 从 `ppi_selected.pt` 还原 pattern 的节点和原始边
- 从 `neg_protein_protein.csv` 找出 pattern 节点之间命中的负边
- 用红色虚线画这些负边，并输出负边两个端点的 `index`


In [ ]:
import os
from itertools import combinations

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch_geometric.data import InMemoryDataset

CURRENT_DIR = os.path.abspath('.')
if os.path.basename(CURRENT_DIR) != 'enumeration-discovery':
    CURRENT_DIR = os.path.join(CURRENT_DIR, 'enumeration-discovery')

MATCH_COUNTS_CSV = os.path.join(CURRENT_DIR, 'processed', 'ppi', 'selected_match_counts.csv')
SELECTED_PT = os.path.join(CURRENT_DIR, 'processed', 'ppi', 'ppi_selected.pt')
NEG_EDGE_CSV = os.path.join(CURRENT_DIR, 'data', 'neg_protein_protein.csv')

MIN_SUPPORT = 100
MAX_PATTERNS_TO_SHOW = None

MATCH_COUNTS_CSV, SELECTED_PT, NEG_EDGE_CSV

In [ ]:
class SelectedPPIDataset(InMemoryDataset):
    def __init__(self, data_path):
        self.data_path = data_path
        super().__init__(root=os.path.dirname(data_path))
        self.data, self.slices = torch.load(data_path)

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return [os.path.basename(self.data_path)]

    def download(self):
        return

    def process(self):
        return


def to_original_edge_list(data):
    orig_ids = data.orig_node_ids.tolist()
    edge_index = data.edge_index

    undirected_edges = []
    seen = set()
    for eid in range(edge_index.size(1)):
        src = int(edge_index[0, eid])
        dst = int(edge_index[1, eid])
        orig_src = int(orig_ids[src])
        orig_dst = int(orig_ids[dst])
        key = tuple(sorted((orig_src, orig_dst)))
        if orig_src == orig_dst or key in seen:
            continue
        seen.add(key)
        undirected_edges.append((orig_src, orig_dst))

    undirected_edges.sort()
    return undirected_edges


def build_pattern_graph(data):
    graph = nx.Graph()
    orig_node_ids = [int(v) for v in data.orig_node_ids.tolist()]
    center_orig_id = int(data.center_orig_id.item()) if hasattr(data, 'center_orig_id') else None

    for node_id in orig_node_ids:
        graph.add_node(node_id)

    positive_edges = to_original_edge_list(data)
    for src, dst in positive_edges:
        graph.add_edge(int(src), int(dst))

    return graph, positive_edges, orig_node_ids, center_orig_id


def load_negative_edge_set(path):
    neg_df = pd.read_csv(path)
    neg_df.columns = [str(c).strip().lower() for c in neg_df.columns]
    if 'src' not in neg_df.columns or 'dst' not in neg_df.columns:
        raise ValueError(f'Expected src,dst columns in {path}')

    neg_edges = set()
    for _, row in neg_df.iterrows():
        src = int(row['src'])
        dst = int(row['dst'])
        if src == dst:
            continue
        neg_edges.add(tuple(sorted((src, dst))))
    return neg_edges


def find_negative_pairs(orig_node_ids, negative_edge_set):
    hit_pairs = []
    for src, dst in combinations(sorted(orig_node_ids), 2):
        key = tuple(sorted((int(src), int(dst))))
        if key in negative_edge_set:
            hit_pairs.append(key)
    return sorted(hit_pairs)


def draw_pattern(graph_index, match_count, graph, positive_edges, negative_pairs, center_orig_id):
    plt.figure(figsize=(8, 6))
    pos = nx.spring_layout(graph, seed=42)

    node_colors = []
    for node in graph.nodes():
        if center_orig_id is not None and int(node) == int(center_orig_id):
            node_colors.append('#d62728')
        else:
            node_colors.append('#4c78a8')

    nx.draw_networkx_nodes(
        graph,
        pos,
        node_color=node_colors,
        node_size=750,
        linewidths=1.0,
        edgecolors='black',
    )
    nx.draw_networkx_labels(
        graph,
        pos,
        labels={node: str(node) for node in graph.nodes()},
        font_size=8,
        font_color='white',
    )

    nx.draw_networkx_edges(
        graph,
        pos,
        edgelist=positive_edges,
        width=2.0,
        edge_color='#7f7f7f',
    )

    if negative_pairs:
        nx.draw_networkx_edges(
            graph,
            pos,
            edgelist=negative_pairs,
            width=2.2,
            edge_color='#d62728',
            style='dashed',
            connectionstyle='arc3,rad=0.12',
        )

    plt.title(
        f'Pattern {graph_index} | support={match_count} | '
        f'nodes={graph.number_of_nodes()} | positive_edges={len(positive_edges)} | '
        f'negative_pairs={len(negative_pairs)}',
        fontsize=11,
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
match_df = pd.read_csv(MATCH_COUNTS_CSV)
match_df.columns = [str(c).strip() for c in match_df.columns]
selected_rows = match_df[match_df['match_count'] >= MIN_SUPPORT].copy()
selected_rows = selected_rows.sort_values(['match_count', 'graph_index'], ascending=[False, True])

if MAX_PATTERNS_TO_SHOW is not None:
    selected_rows = selected_rows.head(MAX_PATTERNS_TO_SHOW)

dataset = SelectedPPIDataset(SELECTED_PT)
negative_edge_set = load_negative_edge_set(NEG_EDGE_CSV)

display(Markdown(f'**Loaded {len(match_df)} match rows**'))
display(Markdown(f'**Showing {len(selected_rows)} patterns with support >= {MIN_SUPPORT}**'))
selected_rows.head()

In [ ]:
for _, row in selected_rows.iterrows():
    graph_index = int(row['graph_index'])
    match_count = int(row['match_count'])

    data = dataset.get(graph_index)
    graph, positive_edges, orig_node_ids, center_orig_id = build_pattern_graph(data)
    negative_pairs = find_negative_pairs(orig_node_ids, negative_edge_set)

    display(Markdown(f'## Pattern {graph_index}'))
    display(Markdown(f'- support(match_count): {match_count}'))
    display(Markdown(f'- center_orig_id: {center_orig_id}'))
    display(Markdown(f'- orig_node_ids: {orig_node_ids}'))
    display(Markdown(f'- negative_edge_pairs(index): {negative_pairs if negative_pairs else []}'))

    draw_pattern(
        graph_index=graph_index,
        match_count=match_count,
        graph=graph,
        positive_edges=positive_edges,
        negative_pairs=negative_pairs,
        center_orig_id=center_orig_id,
    )